In [2]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()


INPUT_FILE = "dedup_ready_batch_50.jsonl"


print("[1/2] Uploading JSONL...")

with open(INPUT_FILE, "rb") as f:
    batch_file = client.files.create(
        file=f,
        purpose="batch"
    )

print(f"[UPLOAD] File ID: {batch_file.id}")


print("[2/2] Creating Batch...")

batch = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)

print(f"[BATCH] Batch ID: {batch.id}")
print(f"[BATCH] Status: {batch.status}")

[1/2] Uploading JSONL...
[UPLOAD] File ID: file-SoEvLJ5xTwz5TgjE2g1f8t
[2/2] Creating Batch...
[BATCH] Batch ID: batch_6a82f17f42a08190b9ac626a83fa7cc6
[BATCH] Status: validating


In [5]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()

# Use the batch ID returned when you submitted the batch
BATCH_ID = batch.id

current_batch = client.batches.retrieve(BATCH_ID)

print("Batch ID:", current_batch.id)
print("Status:", current_batch.status)

if current_batch.request_counts:
    print("Total requests:", current_batch.request_counts.total)
    print("Completed:", current_batch.request_counts.completed)
    print("Failed:", current_batch.request_counts.failed)

if current_batch.output_file_id:
    print("Output file ID:", current_batch.output_file_id)

if current_batch.error_file_id:
    print("Error file ID:", current_batch.error_file_id)

Batch ID: batch_6a82f17f42a08190b9ac626a83fa7cc6
Status: completed
Total requests: 150
Completed: 150
Failed: 0
Output file ID: file-Kgow6VKcWQkQv95btjx5X8


In [6]:
OUTPUT_FILE = "dedup_batch_results_50.jsonl"

if current_batch.status != "completed":
    print(f"Batch is not finished yet. Current status: {current_batch.status}")
else:
    if current_batch.output_file_id is None:
        print("Batch completed but no output file was found.")
    else:
        print(f"Downloading output file: {current_batch.output_file_id}")

        result = client.files.content(
            current_batch.output_file_id
        )

        with open(OUTPUT_FILE, "wb") as f:
            f.write(result.read())

        print(f"[DONE] Results saved to: {OUTPUT_FILE}")

[DONE] Results saved to: dedup_batch_results_50.jsonl
